In [2]:
import torch
import numpy as np
from decord import VideoReader, cpu
from transformers import AutoVideoProcessor, AutoModel

# 1. Cấu hình thiết bị (Ưu tiên GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sử dụng thiết bị: {device}")

# 2. Tải mô hình V-JEPA 2 (biến thể Large) và Processor từ Hugging Face
# Bạn có thể chọn bản "vitl" (300M) hoặc "vitg" (1.2B) [5, 4]
model_id = "facebook/vjepa2-vitl-fpc64-256"
model = AutoModel.from_pretrained(model_id).to(device)
processor = AutoVideoProcessor.from_pretrained(model_id)

# 3. Đọc video bằng decord
# VideoReader hỗ trợ cả đường dẫn cục bộ và URL trực tiếp
video_path = "D:\\College Stuff\\Code\\4th\\ML\\v-jepa 2\\eat test 256.mp4"
vr = VideoReader(video_path, ctx=cpu(0))

# 4. Lấy mẫu 64 khung hình đều nhau xuyên suốt video 
# V-JEPA 2 yêu cầu chính xác số lượng khung hình theo cấu hình pre-train
indices = np.linspace(0, len(vr) - 1, 64, dtype=int)
video_data = vr.get_batch(indices).asnumpy()  # Kết quả trả về: (64, H, W, 3)

# 5. Chuyển đổi định dạng để phù hợp với Processor
# decord trả về (Time, Height, Width, Channels), cần chuyển sang (T, C, H, W)
video_tensor = torch.from_numpy(video_data).permute(0, 3, 1, 2)

# 6. Tiền xử lý (Resizing & Normalization) và Chạy Inference
# FIX: processor nhận numpy array dạng (T, H, W, C) — dùng video_data, không phải video_tensor
inputs = processor(list(video_data), return_tensors="pt").to(device)

with torch.no_grad():
    # Trích xuất đặc trưng ẩn (Embeddings)
    # get_vision_features sẽ bỏ qua bộ dự đoán (predictor) để lấy đặc trưng thuần 
    outputs = model.get_vision_features(**inputs)

print(f"Kích thước đầu ra của đặc trưng: {outputs.shape}")
# Output: (batch_size, sequence_length, hidden_size)


Sử dụng thiết bị: cuda


Loading weights: 100%|██████████| 587/587 [00:00<00:00, 5430.62it/s]


Kích thước đầu ra của đặc trưng: torch.Size([1, 8192, 1024])


In [ ]:
from transformers import AutoVideoProcessor, AutoModelForVideoClassification
import torch
import numpy as np

# Tải phiên bản đã được huấn luyện sẵn trên Something-Something v2
model_id = "facebook/vjepa2-vitl-fpc16-256-ssv2" 
model = AutoModelForVideoClassification.from_pretrained(model_id).to(device)
processor = AutoVideoProcessor.from_pretrained(model_id)

# FIX: Mô hình fpc16 cần đúng 16 khung hình — lấy mẫu lại từ video_data (64 khung)
# indices_16 = np.linspace(0, len(video_data) - 1, 16, dtype=int)
# video_data_16 = video_data[indices_16]  # (16, H, W, 3)

inputs = processor(list(video_data), return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    
# Lấy nhãn hành động có xác suất cao nhất
predicted_class_idx = torch.argmax(logits, dim=-1).item()
print(f"Hành động dự đoán: {model.config.id2label[predicted_class_idx]}")


Loading weights: 100%|██████████| 652/652 [00:00<00:00, 8974.33it/s]


Hành động dự đoán: Scooping [something] up with [something]


In [ ]:
import torch

# Nạp bộ mã hóa và bộ dự đoán hành động (bản Giant 1.2B)
encoder, ac_predictor = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_ac_vit_giant')

encoder.to(device).eval()
ac_predictor.to(device).eval()

# 1. Trích xuất đặc trưng hình ảnh hiện tại (z_k)
# (Giả sử video_tensor đã được xử lý từ decord như ở ví dụ trước)
with torch.no_grad():
    # FIX: torch.hub encoder nhận đầu vào dạng (B, C, T, H, W) — float, chuẩn hoá về [0, 1]
    # video_tensor có dạng (T, C, H, W) → cần permute sang (C, T, H, W) rồi unsqueeze batch
    encoder_input = video_tensor.permute(1, 0, 2, 3).unsqueeze(0).float().div(255.0).to(device)
    z_k = encoder(encoder_input)

    # 4. Giả lập trạng thái robot (7 chiều: x, y, z, euler_x, euler_y, euler_z, gripper)
    # s_k: trạng thái hiện tại, a_k: hành động thực hiện
    s_k = torch.zeros(1, 32, 7).to(device)
    a_k = torch.zeros(1, 32, 7).to(device)

    # 5. Dự đoán trạng thái ẩn tiếp theo bằng ac_predictor
    # Predictor của bản Hub nhận vào 3 tham số trực tiếp: (embeddings, states, actions) 
    z_next_pred = ac_predictor(z_k, s_k, a_k)

print(f"Trích xuất thành công z_k: {z_k.shape}")
print(f"Dự đoán thành công tương lai z_next: {z_next_pred.shape}")

Using cache found in C:\Users\sangg/.cache\torch\hub\facebookresearch_vjepa2_main


Trích xuất thành công z_k: torch.Size([1, 8192, 1408])
Dự đoán thành công tương lai z_next: torch.Size([1, 8192, 1408])
